# DataTrust — Recomendaciones de gobernanza

## Contexto

Los notebooks 01, 02 y 03 auditaron `listings`, `reviews` y `calendar` por separado, con 14 hallazgos identificados y verificados en total (8 en `listings`, 4 en `reviews`, 2 en `calendar`). Este notebook los consolida en una única tabla de gobernanza: no repite el código de la auditoría, cita sus resultados ya verificados y añade una capa de decisión.

## Objetivo de este notebook

Para cada hallazgo: qué riesgo real supone dejarlo sin tratar, qué decisión se propone, y por qué. Es el paso entre "auditar" y "curar" en el framework DataTrust (auditar → priorizar → curar → comparar negocio → comparar modelo). Aquí solo se decide qué hacer, la ejecución de la curación ocurre en el notebook de tratamiento.

## Nota metodológica: qué significa "Prioridad" aquí

La prioridad no es una puntuación estadística ni un estándar externo (no existe una fórmula que combine severidad × alcance × impacto de forma objetiva sin introducir juicios subjetivos disfrazados de números). Es una clasificación cualitativa (**Alta / Media / Baja**) basada en el tipo de problema y, sobre todo, en la consecuencia descrita en la columna "Riesgo". Cada prioridad se acompaña de una justificación explícita en texto, no de un cálculo.

Importante: esta prioridad mide **urgencia de tratamiento de datos**, no relevancia narrativa para el informe de negocio. Un hallazgo puede tener prioridad de tratamiento baja (no requiere una acción de limpieza propia) y aun así ser muy relevante para el informe (por ejemplo, L6, el patrón de licencias no clasificables en anfitriones profesionales, es el hallazgo más citado en la Sección 3 de `listings`, pero no genera una acción de curación distinta de la ya decidida para L3).

La prioridad puede depender además del uso previsto de los datos: un mismo problema puede tener consecuencias diferentes según el análisis o modelo en el que se utilice la tabla, esta tabla de gobernanza es un punto de partida razonado, no una clasificación definitiva e inamovible.

Para la columna "Decisión propuesta" se usa un vocabulario de referencia, no cerrado: *mantener sin tratamiento*, *imputar*, *excluir del análisis*, *reformatear/limpiar*, *revisar manualmente*, *eliminar*, *investigar antes de modificar*.

## Tabla `listings` (8 hallazgos: L1-L5 problemas de calidad, L6-L8 patrones derivados)

| # | Problema | Riesgo | Prioridad | Decisión propuesta | Justificación |
|---|---|---|---|---|---|
| L1 | 12 columnas 100% vacías | No aportan ninguna información utilizable; incluirlas en el modelo o el dashboard no añade valor, solo ruido. | **Alta** | Eliminar estas 12 columnas del dataset curado. | Al ser 100% nulas no existe estrategia de imputación razonable, no hay ningún dato del que partir. Mantenerlas no tiene ningún beneficio. |
| L2 | Nulos parciales en price/bedrooms/bathrooms/beds/host_about/host_location (12,7%-31,8%) | Si se ignoran, cualquier análisis o modelo que use estas columnas pierde filas o se sesga según cómo se traten. | **Media** | Investigar antes de modificar: evaluar imputación específica por columna en el tratamiento (ej. price por mediana de zona/tipo, host_about/host_location con categoría "desconocido"). | Cada columna tiene naturaleza distinta (numérica vs. texto de perfil), aplicar una única estrategia genérica a las 6 sería una decisión poco fundamentada. |
| L3 | Campo `license` no verificable, sin dato + no clasificable (21,1%) | Cualquier análisis de cumplimiento normativo o segmentación por licencia pierde fiabilidad para una quinta parte del inventario. | **Alta** | Mantener sin tratamiento de imputación: conservar las 5 categorías (`license_categoria`) como variable de gobernanza, documentar la limitación en el informe. | No hay información externa en el dataset para reconstruir una licencia real a partir de "Sin dato" u "Otro/no clasificable"; imputar aquí sería inventar datos, no corregir un error. |
| L4 | Licencias con 2 números concatenados + HTML sin limpiar (5.470 registros) | Si se usa `license` como texto libre para extraer patrones automáticamente, estos registros pueden confundir el parseo (contar de más o fallar). | **Media** | Reformatear/limpiar: quitar HTML y separar los múltiples números HUTB en una columna adicional en el tratamiento. | Es un problema de formato solucionable con reglas de texto, la información ya está presente, solo mal estructurada, no requiere inventar nada. |
| L5 | `minimum_nights` > 365 días (3 alojamientos) | Podría distorsionar estadísticas agregadas de estancia mínima si se asume "todo es corta estancia". | **Baja** | Revisar manualmente esos 3 casos, no eliminar automáticamente. | Volumen mínimo (3 de 15.293) y sin evidencia de error: podría ser un alquiler de larga duración legítimo (Airbnb sí ofrece esa modalidad, como se discutió en la revisión). |
| L6 | Anfitriones profesionales sobrerrepresentados en el nuevo registro NRUA, adopción más rápida de la normativa vigente desde jul. 2025 (65,7% vs 18-25%); el residuo genuino de "no clasificable" es 100% particular | Un análisis de cumplimiento por tipo de operador que ignore este patrón podría malinterpretar el ritmo de adaptación normativa por tipo de operador. | **Baja** | No requiere tratamiento de datos adicional (ya cubierto por la decisión de L3); se traslada como hallazgo explícito al informe de negocio. | Es un patrón derivado de L3, no introduce una acción de limpieza propia. Prioridad de tratamiento baja no equivale a poca relevancia narrativa (ver nota metodológica). |
| L7 | Problema de licencia varía por distrito (16,1% Eixample a 33,1% Nou Barris) | Un análisis por distrito que ignore esta variación podría atribuir diferencias de negocio a factores equivocados. | **Baja** | No requiere tratamiento adicional; documentar como contexto geográfico en el análisis de negocio. | Igual que L6, es un desglose de L3, no un problema de calidad nuevo que necesite su propia decisión de curación. |
| L8 | Completitud de host_about/host_location desigual por tipo de anfitrión | Si se imputa host_about/host_location con una única estrategia global, se podría introducir un sesgo distinto por grupo de anfitrión. | **Baja** | Si se decide imputar en L2, considerar `host_profesional` como variable relevante para esa estrategia, no como acción separada. | Es información que mejora la decisión de L2, no un hallazgo independiente que requiera tratamiento propio. |

## Tabla `reviews` (4 hallazgos)

| # | Problema | Riesgo | Prioridad | Decisión propuesta | Justificación |
|---|---|---|---|---|---|
| R1 | Comentarios con texto idéntico a otra reseña (48.302) | Si se usa el texto de reviews para NLP (ej. análisis de sentimiento), la repetición podría sesgar frecuencias de palabras. | **Baja** | Mantener sin tratamiento en esta fase; filtrar duplicados exactos solo si se hace análisis de texto en el futuro. | La mayoría son frases cortas genéricas (verificado en la Sección 4: mediana de 14 caracteres), sin evidencia fuerte de problema. El proyecto además evita deliberadamente que las reseñas sean protagonistas. |
| R2 | Reseñas huérfanas (6.592, 0,64%) | No pueden asociarse con atributos de `listings` (distrito, precio, tipo de anfitrión); cualquier agregación cruzada sin filtrarlas arrastra el problema. | **Alta** | Excluir estas filas de cualquier análisis o cálculo que requiera un `JOIN` con `listings`. | No existe forma de recuperar el alojamiento correspondiente; excluir es la única decisión defendible sin inventar un `listing_id`. |
| R3 | Mismo `reviewer_id` con texto idéntico en alojamientos distintos (512, en 452 combinaciones) | Usarlo como señal de autenticidad sin contexto podría marcar erróneamente reseñas legítimas como sospechosas. | **Baja** | Mantener sin tratamiento automático; documentar como señal para una eventual revisión manual de un equipo de confianza y seguridad, fuera del alcance de este proyecto. | Ya se comprobó que la intensidad es baja en la mayoría de casos (91,4% en solo 2 alojamientos, texto corto). Eliminar sin evidencia adicional no sería defendible. |
| R4 | Reseñas con menos de 5 caracteres (9.917) | Aportan poco valor para análisis de contenido/NLP, pero no representan datos incorrectos. | **Baja** | Mantener sin eliminar de la tabla completa; excluir solo de análisis de texto específicos si se hicieran en el futuro. | No hay base para considerarlas datos erróneos, solo menos informativas. Eliminarlas de la tabla perdería información real (siguen contando en el volumen de reseñas de un alojamiento). |

## Tabla `calendar` (2 hallazgos)

| # | Problema | Riesgo | Prioridad | Decisión propuesta | Justificación |
|---|---|---|---|---|---|
| C1 | Filas huérfanas (41.245, 113 listings, 0,73%) | No pueden vincularse con atributos de `listings` (precio, distrito, tipo de anfitrión); cualquier cálculo de ocupación/ingresos por distrito o tipo que use `calendar` sin filtrarlas arrastra el problema. | **Alta** | Excluir estos 113 `listing_id` (y sus 41.245 filas) de cualquier análisis o cálculo agregado que requiera atributos de `listings`. | Igual que R2: no hay forma de recuperar los atributos correspondientes. `calendar` es además la tabla más grande, con más peso en cualquier cálculo de ocupación. |
| C2 | `minimum_nights` > `maximum_nights` (746 filas, 6 listings) | Representa una configuración imposible que podría romper cualquier lógica de negocio basada en el rango de noches permitido. | **Alta** | Revisar manualmente los 6 alojamientos afectados; dado que están concentrados (no dispersos), es manejable caso por caso, no requiere una regla genérica masiva. | Es una contradicción lógica real ("valor imposible" según el marco de tres niveles usado en todo el proyecto), no una interpretación subjetiva. El volumen concentrado permite revisión dirigida. |

## Resumen cuantificado de prioridades

Los niveles de prioridad asignados arriba se cuentan aquí en código, para que el resumen no sea una afirmación suelta sino un conteo verificable sobre los mismos 14 hallazgos de las tres tablas.

In [1]:
import pandas as pd

hallazgos = pd.DataFrame([
    {"tabla": "listings", "id": "L1", "prioridad": "Alta"},
    {"tabla": "listings", "id": "L2", "prioridad": "Media"},
    {"tabla": "listings", "id": "L3", "prioridad": "Alta"},
    {"tabla": "listings", "id": "L4", "prioridad": "Media"},
    {"tabla": "listings", "id": "L5", "prioridad": "Baja"},
    {"tabla": "listings", "id": "L6", "prioridad": "Baja"},
    {"tabla": "listings", "id": "L7", "prioridad": "Baja"},
    {"tabla": "listings", "id": "L8", "prioridad": "Baja"},
    {"tabla": "reviews", "id": "R1", "prioridad": "Baja"},
    {"tabla": "reviews", "id": "R2", "prioridad": "Alta"},
    {"tabla": "reviews", "id": "R3", "prioridad": "Baja"},
    {"tabla": "reviews", "id": "R4", "prioridad": "Baja"},
    {"tabla": "calendar", "id": "C1", "prioridad": "Alta"},
    {"tabla": "calendar", "id": "C2", "prioridad": "Alta"},
])

print("Total de hallazgos:", len(hallazgos))
print()
print("Por tabla:")
print(hallazgos["tabla"].value_counts())
print()
print("Por prioridad:")
print(hallazgos["prioridad"].value_counts())

Total de hallazgos: 14

Por tabla:
tabla
listings    8
reviews     4
calendar    2
Name: count, dtype: int64

Por prioridad:
prioridad
Baja     7
Alta     5
Media    2
Name: count, dtype: int64


**Interpretación**: de los 14 hallazgos, 5 tienen prioridad Alta (L1, L3, R2, C1, C2), 2 Media (L2, L4) y 7 Baja (L5, L6, L7, L8, R1, R3, R4). Un patrón consistente en las prioridades Altas: **3 de las 5 son problemas de integridad referencial o estructura (L1, R2, C1)** y **2 son contradicciones lógicas internas (L3 por su papel de caso estrella de gobernanza, C2 por ser un valor imposible)**. Los patrones de negocio derivados (L6, L7, L8) son consistentemente Baja en tratamiento de datos, aunque como se explicó en la nota metodológica, eso no reduce su relevancia narrativa para el informe.

Esta tabla pasa al notebook de tratamiento y curación, donde las decisiones propuestas aquí se ejecutan en código sobre el dataset raw para producir el dataset curado.